# Email Spam Classification
### End-to-End ML Pipeline: TF-IDF + Naive Bayes + Logistic Regression

**Author:** Mohit Masavarapu  
**Stack:** Python, Scikit-learn, Pandas, Matplotlib, Seaborn

---

## Pipeline Overview
1. Data Loading & Exploration
2. Data Cleaning & Preprocessing
3. Feature Extraction (TF-IDF)
4. Model Training (Naive Bayes + Logistic Regression)
5. Hyperparameter Tuning (GridSearchCV)
6. Evaluation (Accuracy, F1, ROC-AUC, Confusion Matrix)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries loaded successfully!')

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('../data/spam.csv')
print(f'Shape: {df.shape}')
print(f'\nClass Distribution:\n{df["label"].value_counts()}')
df.head(10)

In [ ]:
# Visualise class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts()
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%', colors=['#55A868', '#C44E52'], startangle=90)
axes[0].set_title('Class Distribution')

df['message_length'] = df['message'].str.len()
df.groupby('label')['message_length'].plot(kind='hist', alpha=0.6, bins=40, ax=axes[1],
                                            color=['#55A868', '#C44E52'], legend=True)
axes[1].set_title('Message Length Distribution by Class')
axes[1].set_xlabel('Message Length (characters)')
plt.tight_layout()
plt.show()

## 2. Feature Extraction with TF-IDF

**TF-IDF (Term Frequency-Inverse Document Frequency)** converts text into numerical features by:
- **TF**: How frequently a word appears in a document
- **IDF**: How rare the word is across all documents

Words that are common in spam (like "FREE", "WIN", "CLAIM") but rare in ham get high TF-IDF scores.

In [ ]:
X = df['message']
y = df['label'].map({'ham': 0, 'spam': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english', sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Train shape: {X_train_tfidf.shape}')
print(f'Test  shape: {X_test_tfidf.shape}')

## 3. Model Training

In [ ]:
# Naive Bayes
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train_tfidf, y_train)
nb_pred = nb.predict(X_test_tfidf)
nb_acc  = accuracy_score(y_test, nb_pred)

# Logistic Regression
lr = LogisticRegression(C=5.0, max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
lr_pred = lr.predict(X_test_tfidf)
lr_acc  = accuracy_score(y_test, lr_pred)

print(f'Naive Bayes Accuracy       : {nb_acc:.4f} ({nb_acc*100:.2f}%)')
print(f'Logistic Regression Accuracy: {lr_acc:.4f} ({lr_acc*100:.2f}%)')

## 4. Hyperparameter Tuning

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', sublinear_tf=True)),
    ('clf',   LogisticRegression(max_iter=1000, random_state=42))
])

param_grid = {
    'tfidf__max_features': [3000, 5000],
    'tfidf__ngram_range':  [(1, 1), (1, 2)],
    'clf__C':              [1.0, 5.0, 10.0],
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print(f'Best params: {grid.best_params_}')
print(f'Best CV score: {grid.best_score_:.4f}')

## 5. Final Evaluation

In [ ]:
best_model = grid.best_estimator_
best_pred  = best_model.predict(X_test)
best_acc   = accuracy_score(y_test, best_pred)

print(f'Test Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)')
print()
print(classification_report(y_test, best_pred, target_names=['Ham', 'Spam']))

cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='accuracy')
print(f'5-Fold CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.title('Confusion Matrix — Tuned Logistic Regression')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 6. Live Prediction Demo

In [ ]:
test_messages = [
    "Hey, are you coming to the meeting today?",
    "WINNER!! You've been selected for a £900 prize! Call 09061701461 NOW!",
    "Can you pick up some milk on the way home?",
    "FREE ringtones! Text RING to 87777 to get yours now. 150p/msg.",
    "The project deadline is next Friday.",
]

preds = best_model.predict(test_messages)
probs = best_model.predict_proba(test_messages)

for msg, pred, prob in zip(test_messages, preds, probs):
    label = 'SPAM 🚫' if pred == 1 else 'HAM  ✅'
    conf  = prob[pred]
    print(f'[{label}] ({conf:.2%}) — {msg[:60]}')